In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sbn


In [28]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [29]:
train.shape, test.shape

((891, 12), (418, 11))

In [30]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [31]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [32]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB


In [33]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    str    
 3   Sex          418 non-null    str    
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    str    
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     str    
 10  Embarked     418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 52.8 KB


In [34]:
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [35]:
test.isnull().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [36]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

def make_features(df):
    df = df.copy()

    df["Title"] = (
        df["Name"]
        .str.extract(r",\s*([^.]+)\.")[0]
        .str.strip()
    )

    common_titles = ["Mr", "Miss", "Mrs", "Master"]
    df["Title"] = df["Title"].where(
        df["Title"].isin(common_titles),
        "Rare"
    )

    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["SexPclass"] = df["Sex"] + "_Pclass_" + df["Pclass"].astype(str)
    df["HasCabin"] = df["Cabin"].notna().astype(int)

    return df.drop(
        columns=["PassengerId", "Name", "Ticket", "Cabin"]
    )

In [37]:
X = make_features(train.drop(columns="Survived"))
y = train["Survived"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [38]:
numeric_features = [
    "Pclass", "Age", "SibSp", "Parch",
    "Fare", "FamilySize", "IsAlone", "HasCabin"
]

categorical_features = [
    "Sex", "Embarked", "Title", "SexPclass"
]

In [39]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [40]:
from sklearn.tree import DecisionTreeClassifier

tree_model = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(max_depth=3, random_state=42)),
    ]
)

In [41]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(max_depth=3, random_state=42)),
    ]
    
)

In [42]:
from sklearn.base import clone
from sklearn.metrics import confusion_matrix

# Fit a separate copy using only the training split
validation_model = clone(model)
validation_model.fit(X_train, y_train)
valid_predictions = validation_model.predict(X_valid)

print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_valid, valid_predictions))

# Attach predictions to the original validation rows for inspection
validation_errors = train.loc[
    X_valid.index,
    ["PassengerId", "Survived", "Sex", "Pclass", "Age",
     "Fare", "SibSp", "Parch", "Cabin"]
].copy()

validation_errors["Predicted"] = valid_predictions
validation_errors["ErrorType"] = np.select(
    [
        (validation_errors["Survived"] == 0) & (validation_errors["Predicted"] == 1),
        (validation_errors["Survived"] == 1) & (validation_errors["Predicted"] == 0),
    ],
    ["False positive", "False negative"],
    default="Correct"
)

mistakes = validation_errors[
    validation_errors["ErrorType"] != "Correct"
].sort_values(["ErrorType", "Pclass", "Sex"])

mistakes

Confusion matrix [[TN, FP], [FN, TP]]:
[[100  10]
 [ 19  50]]


,PassengerId,Survived,Sex,Pclass,Age,Fare,SibSp,Parch,Cabin,Predicted,ErrorType
712,713,1,male,1,48.0,52.0000,1,0,C126,0,False negative
701,702,1,male,1,35.0,26.2875,0,0,E24,0,False negative
447,448,1,male,1,34.0,26.5500,0,0,NaN,0,False negative
690,691,1,male,1,31.0,57.0000,1,0,B20,0,False negative
55,56,1,male,1,NaN,35.5000,0,0,C52,0,False negative
23,24,1,male,1,28.0,35.5000,0,0,A6,0,False negative
607,608,1,male,1,27.0,30.5000,0,0,NaN,0,False negative
587,588,1,male,1,60.0,79.2000,1,1,B41,0,False negative
17,18,1,male,2,NaN,13.0000,0,0,NaN,0,False negative
570,571,1,male,2,62.0,10.5000,0,0,NaN,0,False negative


In [43]:
mistakes.groupby(["ErrorType", "Sex", "Pclass"]).size().sort_values(ascending=False)

ErrorType       Sex     Pclass
False negative  male    1         8
                        3         6
False positive  female  3         5
False negative  female  3         3
False positive  male    1         3
False negative  male    2         2
False positive  female  1         1
                        2         1
dtype: int64

In [44]:
rate_data = validation_errors.assign(
    actual_survivor=validation_errors["Survived"].eq(1),
    false_negative=(
        validation_errors["Survived"].eq(1)
        & validation_errors["Predicted"].eq(0)
    ),
    actual_death=validation_errors["Survived"].eq(0),
    false_positive=(
        validation_errors["Survived"].eq(0)
        & validation_errors["Predicted"].eq(1)
    ),
)

by_sex = rate_data.groupby("Sex").agg(
    actual_survivors=("actual_survivor", "sum"),
    false_negatives=("false_negative", "sum"),
    actual_deaths=("actual_death", "sum"),
    false_positives=("false_positive", "sum"),
)

by_sex["false_negative_rate_%"] = (
    100 * by_sex["false_negatives"] / by_sex["actual_survivors"]
).round(1)

by_sex["false_positive_rate_%"] = (
    100 * by_sex["false_positives"] / by_sex["actual_deaths"]
).round(1)

by_sex

,actual_survivors,false_negatives,actual_deaths,false_positives,false_negative_rate_%,false_positive_rate_%
Sex,,,,,,
female,45,3,16,7,6.7,43.8
male,24,16,94,3,66.7,3.2


In [45]:
model.fit(X_train, y_train)

validation_accuracy = model.score(X_valid, y_valid)
validation_accuracy

0.8379888268156425

In [46]:
X_test = make_features(test)

predictions = model.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": predictions
})

submission.to_csv("submission.csv", index=False)

## Evaluation and Kaggle result

The current holdout validation accuracy is `0.8324`. The submitted model scored `0.77511` on Kaggle's hidden test labels. `test.csv` does not contain the true `Survived` values, so its local test accuracy cannot be calculated; Kaggle provides that score after submission.

The preprocessing pipeline is shared across training, validation, and test data. Imputers and encoders are fitted only on the training data, which prevents preprocessing leakage and keeps the datasets consistent.

In [47]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5, 
    shuffle=True, 
    random_state=42)

In [48]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Fold accuracies:", cv_scores)
print(f"Mean CV accuracy: {cv_scores.mean():.4f}")
print(f"CV standard deviation: {cv_scores.std():.4f}")

Fold accuracies: [0.82122905 0.83707865 0.8258427  0.84269663 0.84269663]
Mean CV accuracy: 0.8339
CV standard deviation: 0.0088


In [49]:
tree_scores = cross_val_score(
    tree_model, 
    X, 
    y, 
    cv=cv, 
    scoring="accuracy")

print("Fold accuracies:", tree_scores)
print(f"Mean CV accuracy: {tree_scores.mean():.4f}")
print(f"CV standard deviation: {tree_scores.std():.4f}")

Fold accuracies: [0.82122905 0.82022472 0.78089888 0.80898876 0.83707865]
Mean CV accuracy: 0.8137
CV standard deviation: 0.0187


In [50]:
random_forest_scores = cross_val_score(
    random_forest_model, 
    X, 
    y, 
    cv=cv, 
    scoring="accuracy")

print("Fold accuracies:", random_forest_scores)
print(f"Mean CV accuracy: {random_forest_scores.mean():.4f}")
print(f"CV standard deviation: {random_forest_scores.std():.4f}")

Fold accuracies: [0.80446927 0.79213483 0.81460674 0.80337079 0.84269663]
Mean CV accuracy: 0.8115
CV standard deviation: 0.0172


## Final training and future improvements

After choosing the model using validation and cross-validation, retrain it on all labeled training data before generating the final submission. Possible next experiments are Random Forest or Gradient Boosting models and additional features such as ticket-group size.

In [51]:
# Retrain the selected pipeline on every labeled training row.
model.fit(X, y)

final_predictions = model.predict(X_test)

final_submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": final_predictions
})

final_submission.to_csv("submission.csv", index=False)
final_submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


In [52]:
train.assign(Deck=train["Cabin"].str[0]).groupby("Deck")["Survived"].agg(
    count="size", survival_rate="mean"
).sort_values("count", ascending=False)

,count,survival_rate
Deck,,
C,59,0.593220
B,47,0.744681
D,33,0.757576
E,32,0.750000
A,15,0.466667
F,13,0.615385
G,4,0.500000
T,1,0.000000
